## Exercise 6: Shared utility functions, data catalogs

Skills: 
* Import shared utils
* Data catalog
* Use functions to repeat certain data cleaning steps

References: 
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html
* https://docs.calitp.org/data-infra/analytics_tools/data_catalogs.html

In [ ]:
import geopandas as gpd
import intake
import pandas as pd
import matplotlib.pyplot as plt 

from calitp.tables import tbl
from calitp import query_sql
from siuba import *

from shapely.geometry import Polygon, LineString, Point
from shapely import wkt
from shapely.geometry import Point, Polygon
from shapely.wkt import loads

import shared_utils
import altair as alt
from shared_utils import altair_utils 
from shared_utils import styleguide

import branca
import folium

#commas to separate 1000s 
pd.options.display.float_format = '{:,}'.format

## Create a data catalog 

* Include one geospatial data source and one tabular (they should be related...your analysis depends on combining them)
* Import your datasets using the catalog method


In [ ]:
catalog = intake.open_catalog("./Catalog.yml")

In [ ]:
#Importing geojson 
geojson = catalog.geojson_districts.read()

In [ ]:
geojson

In [ ]:
geojson = geojson[['DISTRICT','Shape_Length','Shape_Area','geometry']]

In [ ]:
geojson

In [ ]:
#tircp = catalog.TIRCP_processed.read()

In [ ]:
tircp = pd.read_excel('gs://calitp-analytics-data/data-analyses/tircp/Tableau_Sheet.xlsx')

In [ ]:
#only keep columns I want. Drop NA for district, filter out projects with districts coded as "various"
tircp = (tircp[['Award_Year', 'Local_Agency','District','Local_Agency_City','TIRCP_Amount', 'County','Project_Title']]
         .loc[tircp['District'] != 'Various'].dropna(subset=['District']))

In [ ]:
tircp.shape

In [ ]:
#Renaming districts in geojson file.
tircp['District'] = (tircp['District'].replace({'District 7: Los Angeles': 7,
                                                    'District 2: Redding': 2,
                                            'District 4: Bay Area / Oakland': 4,
                                            'VAR':'Various',
                                           'District 10: Stockton': 10,
                                            'District 11: San Diego': 11,
                                            'District 3: Marysville / Sacramento':3,
                                             'District 12: Orange County':12,
                                            'District 8: San Bernardino / Riverside':8,
                                            'District 5: San Luis Obispo / Santa Barbara':5,
                                            'District 9: Bishop': 9,
                                            'District 6: Fresno / Bakersfield':6,
                                            'District 1: Eureka':1
                                                 }))
    

## Combine datasets 

Tiffany's Notes
* Do a merge or spatial join to combine the geospatial and tabular data
* Create a new column of a summary statistic to visualize
* Rely on `shared_utils` to do at least one operation (aggregation, re-projecting to a different CRS, exporting geoparquet, etc)


* Do left join on geojson, code the missing districts with dummy values so the districts will show up.

In [ ]:
#Joining the 2 dataframes
joined = geojson.merge(tircp, how = 'outer', left_on ='DISTRICT', right_on = 'District', indicator = True)

In [ ]:
#only want rows where merge is both.
joined = joined.loc[joined['_merge'] != 'left_only'] 

In [ ]:
joined

In [ ]:
shared_utils.geography_utils.aggregate_by_geography??

In [ ]:
aggregation_function = shared_utils.geography_utils.aggregate_by_geography(joined, ['District'], 
                                                                           sum_cols = ['TIRCP_Amount'], 
                                                                           nunique_cols =['Project_Title']).sort_values('District')
aggregation_function

### Looking at TIRCP received by district

In [ ]:
#aggregated df to keep geometry.
subset_TIRCP = joined[['District', 'geometry','TIRCP_Amount']]
total_TIRCP_2 = subset_TIRCP.dissolve(by='District', aggfunc='sum').sort_values('TIRCP_Amount').rename(columns = {'TIRCP_Amount':'Total_TIRCP_Received'}).reset_index()
total_TIRCP_2['California'] = 1

In [ ]:
#https://matplotlib.org/2.0.2/users/colormaps.html

In [ ]:
fig, ax = plt.subplots(1, figsize=(9, 9))
ax.axis('off')
ax.set_title('Total TIRCP by District',
             fontdict={'fontsize': '15', 'fontweight' : '3'})
fig = total_TIRCP_2.plot(column='Total_TIRCP_Received', cmap='summer', ax=ax,legend=True)

In [ ]:
choropleth_dict = ({
            "layer_name": 'my layer',
            "MIN_VALUE": int(0),
            "MAX_VALUE": int(2458538000),
            "plot_col_name": 'Total_TIRCP_Received',
            "fig_width": '100%',
            "fig_height": '100%',
            "fig_min_width_px": '600px',
            "fig_min_height_px": '600px'})

In [ ]:
colorscale_TIRCP = branca.colormap.StepColormap(
                colors=["#F6BF16", "#E16B26",  "#00896B"], 
                index=[0, 93_614_000, 351_601_500], 
                vmin=0, vmax=2_458_538_000,
)
colorscale_TIRCP

In [ ]:
REGION_CENTROIDS = shared_utils.map_utils.REGION_CENTROIDS

In [ ]:
shared_utils.map_utils.make_ipyleaflet_choropleth_map(total_TIRCP_2, 
                                                     plot_col = 'Total_TIRCP_Received',
                                                     geometry_col = 'District', 
                                                     choropleth_dict = choropleth_dict, 
                                                     colorscale = colorscale_TIRCP,
                                                     zoom=7, 
                                                     centroid = REGION_CENTROIDS['CA'][0])

### Looking at # of Projects in a District

In [ ]:
#aggregated df to keep geometry for # of projects
subset_projects = joined[['District', 'geometry','Project_Title']]
total_projects_2 = subset_projects.dissolve(by='District', aggfunc='nunique').sort_values('District').rename(columns = {'Project_Title':'Total_Projects'}).reset_index()
total_projects_2

In [ ]:
fig2, ax = plt.subplots(1, figsize=(9, 9))
ax.axis('off')
ax.set_title('Total Projects by District',
             fontdict={'fontsize': '15', 'fontweight' : '3'})
fig2 = total_projects_2.plot(column='Total_Projects', cmap='summer', ax=ax,legend=True)


## Use functions to do parameterized visualizations
* Use a function to create your chart
* Within the function, the colors should use the Cal-ITP theme that is available in `styleguide`
* Within the function, there should be at least 1 parameter that changes (ex: chart title reflects the correct county, legend title reflects the correct county, etc)
* Produce 3 charts, using your function each time, and have the function correctly insert the parameters 

In [ ]:
#Renaming districts from number to string
total_projects_2['District'] = (total_projects_2['District'].replace({7:'District 7: Los Angeles',
                                            4:'District 4: Bay Area / Oakland',
                                            'VAR':'Various',
                                            10:'District 10: Stockton',
                                            11:'District 11: San Diego',
                                            3:'District 3: Marysville / Sacramento',
                                            12: 'District 12: Orange County',
                                            8: 'District 8: San Bernardino / Riverside',
                                            5:'District 5: San Luis Obispo / Santa Barbara',
                                            6:'District 6: Fresno / Bakersfield',
                                            1:'District 1: Eureka'
                                                 }))

total_TIRCP_2['District'] = (total_TIRCP_2['District'].replace({7:'District 7: Los Angeles',
                                            4:'District 4: Bay Area / Oakland',
                                            'VAR':'Various',
                                            10:'District 10: Stockton',
                                            11:'District 11: San Diego',
                                            3:'District 3: Marysville / Sacramento',
                                            12: 'District 12: Orange County',
                                            8: 'District 8: San Bernardino / Riverside',
                                            5:'District 5: San Luis Obispo / Santa Barbara',
                                            6:'District 6: Fresno / Bakersfield',
                                            1:'District 1: Eureka'
                                                 }))

In [ ]:
def labeling(word):
    # Add specific use cases where it's not just first letter capitalized
    LABEL_DICT = { "prepared_y": "Year",
              "dist": "District",
              "nunique":"Number of Unique",
              "project_no": "Project Number"}
    
    if (word == "mpo") or (word == "rtpa"):
        word = word.upper()
    elif word in LABEL_DICT.keys():
        word = LABEL_DICT[word]
    else:
        #word = word.replace('n_', 'Number of ').title()
        word = word.replace('unique_', "Number of Unique ").title()
        word = word.replace('_', ' ').title()
    
    return word


In [ ]:
#object to allow me to input a chart title 
def bar_chart1(df, x_col, y_col, color_col, chart_title=''):
    
    if chart_title == "":
        chart_title = (f"{labeling(x_col)} by {labeling(y_col)}")
    
    chart = (alt.Chart(df)
             .mark_bar()
             .encode(
                 x=alt.X(x_col, title=labeling(x_col)),
                 y=alt.Y(y_col, title=labeling(y_col), sort=('-x')),
                 color = alt.Color(color_col,
                                  scale=alt.Scale(
                                      range=altair_utils.CALITP_CATEGORY_BOLD_COLORS),
                                      legend=alt.Legend(title=(labeling(color_col)))
                                  ))
             .properties( 
                          title=chart_title)
    )
    

    chart=styleguide.preset_chart_config(chart)
    
    return chart

In [ ]:
total_projects_2.dtypes

In [ ]:
bar_chart1(total_projects_2, 'Total_Projects', 'District', 'District')

In [ ]:
bar_chart1(total_TIRCP_2,'Total_TIRCP_Received', 'District', 'District', 'Total TIRCP Funds by District')